## PoI Plotter — VCO Model Exploration & Design Tool

This notebook provides an **interactive visualization and design tool** for the VCO-based conductance sensing system.

Its main purpose is to:

* **Visualize the full signal chain** from $(G, i_{dc}, f_s, D_{AFE}, N)$ to $(\Delta G, P)$
* **Expose all intermediate variables and equations**
* **Understand sensitivities and trade-offs**
* **Solve the inverse problem**: find optimal operating points from target specifications

 The tool combines **forward modeling + reverse optimization** in a single interactive dashboard 

---

## Core Functionality

The plotter allows you to:

### 1. Forward analysis (physics understanding)

Given:

* Conductance $G$
* Bias current $i_{dc}$
* Sampling frequency $f_s$

It computes and visualizes:

* $V_{in}$, $f_{osc}$, $K_{VCO}$
* Frequency error contributions ($\Delta f_{sampling}$, $\Delta f_{adev}$)
* Voltage error $\Delta V_{in}$
* Conductance resolution $\Delta G$
* Power consumption ($P_{idc}, P_{VCO}, P_{CNT}, P_{TOT}$)

 This corresponds to the **physical signal chain** implemented in `forward_compute()` 

---

### 2. Reverse design (control & optimization)

Given:

* Target resolution $\Delta G_{\text{target}}$
* Maximum power $P_{max}$

The tool:

* Sweeps all valid $i_{dc}$
* Computes $(\Delta G, P_{tot})$ for each point
* Builds the feasible region:
  $$
  \Delta G \le \Delta G_{\text{target}}, \quad P_{tot} \le P_{max}
  $$
* Selects the optimal current:
  $$
  i_{dc,opt} = \max(\text{feasible } i_{dc})
  $$

Implemented in `reverse_compute()` 

---

## Interactive Dashboard

The UI is split into two control panels:

### Forward Controls

* **$G$ (μS)**: imposed conductance
* **$i_{dc}$ (μA)**: operating current (bounded automatically)
* **$f_s$ (Hz)**: sampling frequency

### Reverse Controls

* **$\Delta G_{\text{target}}$ (nS)**: desired resolution
* **$P_{max}$ (μW)**: power constraint
* **Variance toggle**: include/exclude Allan deviation

The sliders dynamically adapt (e.g. valid $\Delta G$ range) 

---

## Visualization Panels

The dashboard displays a **6-panel system view**:

### 1. VCO Transfer Curve

* Measured data + model
* Operating point highlighted

### 2. Inputs & Intermediate Values

* $V_{in}$, $f_{osc}$, $K_{VCO}$, $\Delta V_{in}$, $\Delta f_{osc}$

### 3. Frequency Error Breakdown

* Sampling vs Allan deviation contributions

### 4. Output Metrics

* $\Delta G$ (nS)
* Power contributions (μW)

### 5. Trade-off Curve ($i_{dc}$ sweep)

* $\Delta G$ vs $i_{dc}$
* Power vs $i_{dc}$
* Feasible region (green)
* Optimal point (highlighted) 

### 6. Summary Panel

* Current operating point
* Achievable $\Delta G$ range
* Optimal solution (if feasible)

---

## Model Details

### VCO Representation

Two interchangeable models:

* **Polynomial model** (smooth, analytical)
* **LUT model** (data-driven, more accurate)

Controlled via `representation` in `VCO_Model` 

---

### Frequency Error Model

Total frequency uncertainty:
$$
\Delta f_{osc} = \max(\Delta f_{sampling}, \Delta f_{adev})
$$

* **Sampling error**:
  $$
  \Delta f_{sampling} = f_s \cdot \sqrt{avg\_window}
  $$

* **Allan deviation** (optional):

  * extracted from measured oscillator data
  * depends on $V_{in}$ and $\tau = 1/f_s$

---

### Conductance Resolution

Computed from the full nonlinear model:
$$
\Delta G = \left|\frac{\Delta f_{osc} \cdot G^2}{K_{VCO} \cdot i_{dc} + \Delta f_{osc} \cdot G}\right|
$$

---

## How to Use the Tool

### Understanding the system

* Fix $G$, vary $i_{dc}$ and $f_s$
* Observe how $\Delta G$ and power evolve
* Identify optimal regions manually

### Designing the system

* Set:

  * $\Delta G_{\text{target}}$
  * $P_{max}$
* Let the tool:

  * compute feasible region
  * highlight optimal $i_{dc}$

### Key insight

* Increasing $i_{dc}$:

  * improves sensitivity (↓ $\Delta G$)
  * reduces total power (in this regime)
* The optimal point is typically:
    **largest feasible $i_{dc}$**



In [4]:
import sys
import os
from PoI_plotter import *

# Resolve path to vendor VCO model
vendor_vco_path = os.path.abspath(
    os.path.join(os.getcwd(), 
    '../../hw/vendor/analog-library/VCO/VCO_characteristics')
)
sys.path.insert(0, vendor_vco_path)
from vco_model import VCOADCModel

import warnings
warnings.filterwarnings('ignore')

## VCO_Model class

In [5]:
# instantiate the model with our data
VCO_model = VCOADCModel()

In [6]:
PoI_plotter(VCO_model, variance=1)